In [ ]:
from qgravnet import QGravNetFactory

from utils.data import load_processed
from utils.evaluation import load_run
from utils.files import get_project_root_dir

PROJECT_ROOT = get_project_root_dir("hls4ml-gravnet")

In [ ]:
model_cfg, weights_path, history, datapath = load_run(PROJECT_ROOT / "data/results/train1")

D = load_processed(datapath)

trained_model = QGravNetFactory(**model_cfg).create_keras_model(128, 4)
trained_model.load_weights(weights_path)

test_energy_pred, test_pid_pred = trained_model.predict(D["X_hits_test"])

In [ ]:
import hls4ml
from hls4ml_gravnet.hls4ml_extension.gravnet_core import GravNetCore
from hls4ml_gravnet.hls4ml_extension.gravnet_core_handler import GravNetCoreHandler  # noqa: F401
from hls4ml_gravnet.hls4ml_extension.gravnet_core_template import GravNetCoreConfigTemplate, GravNetCoreFunctionTemplate

hls4ml.model.layers.register_layer('GravNetCore', GravNetCore)
backend = hls4ml.backends.get_backend('Vivado')
try:
    backend.register_template(GravNetCoreConfigTemplate)
    backend.register_template(GravNetCoreFunctionTemplate)
    backend.register_source(PROJECT_ROOT / "hls4ml_gravnet" / "hls" / "nnet_gravnet_core.h")
except Exception:
    pass  # Already registered

In [ ]:
hls_config = hls4ml.utils.config_from_keras_model(
    trained_model, granularity='model', backend='Vitis', default_precision='fixed<32,16>'
)
hls_model = hls4ml.converters.convert_from_keras_model(
    trained_model, hls_config=hls_config, output_dir=str(PROJECT_ROOT / "data" / "hls4ml_out" / "toy_calo")
)
hls_model.compile()

In [ ]:
hls_test_energy_pred, hls_test_pid_pred = hls_model.predict(D["X_hits_test"])

In [ ]:
from utils.evaluation import display_evaluation_results

display_evaluation_results(test_energy_pred=test_energy_pred, test_pid_pred=test_pid_pred, D=D, model_cfg=model_cfg)
display_evaluation_results(test_energy_pred=hls_test_energy_pred, test_pid_pred=hls_test_pid_pred, D=D, model_cfg=model_cfg)